# TabPFNv2 — CPU

Dùng phiên bản **TabPFN-2** (Nature 2025), **không dùng model mặc định TabPFN-3/3.5**. Repository: https://github.com/PriorLabs/TabPFN .

**Lưu ý:** Chạy trên CPU, không pretrain từ đầu. Lần chạy đầu cần Internet để cài package/tải checkpoint. Nếu không có package/checkpoint, notebook ghi trạng thái lỗi; **không tự thay bằng mô hình giả lập**.

## 1. Cài thư viện
Chạy lệnh dưới đây trong Terminal hoặc bỏ dấu `#` ở cell tiếp theo. Nên dùng **Python 3.10–3.11** trong môi trường ảo riêng cho mỗi mô hình để tránh xung đột dependency.

In [1]:
%pip install tabpfn scikit-learn pandas torch psutil

  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached matplotlib-3.11.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (80 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached contourpy-1.4.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.66.1-cp314-cp314-macosx_10_15_universal2.whl.metadata (130 kB)
  Using cached kiwisolver-1.5.1-cp314-cp314-macosx_11_0_arm64.whl.metadata (5.2 kB)
  Using cached pillow-12.3.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (9.1 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certi

## 2. Đọc dữ liệu và chia tập thống nhất

In [2]:
from pathlib import Path
import os, json, time, traceback, platform, importlib.metadata
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets' / 'breast_cancer_wisconsin.csv').exists():
    if (ROOT.parent / 'datasets' / 'breast_cancer_wisconsin.csv').exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError('Mở notebook từ thư mục project hoặc chạy: os.chdir(duong_dan_giai_nen_ZIP)')
DATASET = ROOT / 'datasets' / 'breast_cancer_wisconsin.csv'
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
SPLITS = ROOT / 'splits'; SPLITS.mkdir(exist_ok=True)
MODEL_ID = 'TabPFNv2'
SEEDS = [42]                # Sau khi chạy thử: [42, 43, 44, 45, 46]
TRAIN_SIZES = [50, 100]    # Sau khi chạy thử: [50, 100, 200, 300]
N_CPU_THREADS = min(4, os.cpu_count() or 2)
os.environ['CUDA_VISIBLE_DEVICES'] = ''
os.environ['OMP_NUM_THREADS'] = str(N_CPU_THREADS)

frame = pd.read_csv(DATASET)
X = frame.drop(columns=['row_id', 'target']).to_numpy(dtype=np.float32)
y = frame['target'].to_numpy(dtype=np.int64)
assert len(np.unique(y)) == 2 and len(frame) == 569

def indices_for_seed(seed):
    path = SPLITS / f'breast_cancer_seed{seed}.json'
    if path.exists():
        s = json.loads(path.read_text(encoding='utf-8'))
    else:
        ids = np.arange(len(y))
        trainpool, test = train_test_split(ids, test_size=0.2, stratify=y, random_state=seed)
        train, val = train_test_split(trainpool, test_size=0.2, stratify=y[trainpool], random_state=seed)
        s = {'seed': seed, 'train': sorted(map(int,train)), 'val': sorted(map(int,val)), 'test': sorted(map(int,test))}
        path.write_text(json.dumps(s,indent=2),encoding='utf-8')
    assert not (set(s['train']) & set(s['test']))
    assert not (set(s['val']) & set(s['test']))
    assert not (set(s['train']) & set(s['val']))
    return s

def make_fold(seed, n):
    s = indices_for_seed(seed)
    pool = np.asarray(s['train'],dtype=int)
    if n > len(pool):
        raise ValueError(f'n_train={n} > train pool {len(pool)}')
    chosen, _ = train_test_split(pool, train_size=n, stratify=y[pool], random_state=seed + 101)
    chosen = np.sort(chosen)
    return (X[chosen], y[chosen], X[s['val']], y[s['val']], X[s['test']], y[s['test']], chosen, s)

def metrics(y_true, y_pred, prob):
    p = np.asarray(prob,dtype=float)
    assert p.ndim == 2 and p.shape == (len(y_true),2), p.shape
    p = np.clip(p, 1e-8, 1.0); p = p / p.sum(axis=1,keepdims=True)
    return {'accuracy':accuracy_score(y_true,y_pred),
            'macro_f1':f1_score(y_true,y_pred,average='macro',zero_division=0),
            'roc_auc':roc_auc_score(y_true,p[:,1]),
            'log_loss':log_loss(y_true,p,labels=[0,1])}

def record(status, seed, n, fit_sec=None, pred_sec=None, scores=None, err='', version=''):
    out = {'model':MODEL_ID,'dataset':'breast_cancer_wisconsin', 'seed':seed,
           'train_size':n,'status':status,'accuracy':np.nan,'macro_f1':np.nan,
           'roc_auc':np.nan,'log_loss':np.nan,'fit_seconds':fit_sec,
           'predict_seconds':pred_sec,'package_version':version,
           'error':str(err)[:800]}
    if scores: out.update(scores)
    return out

def save_results(rows):
    file=RESULTS / (MODEL_ID.lower().replace(' ','_')+'.csv')
    pd.DataFrame(rows).to_csv(file,index=False)
    print('Đã ghi:',file)
    display(pd.DataFrame(rows))
    return file

print('DATASET:', DATASET, '| shape:',X.shape,'| classes:',np.bincount(y))
print('CPU:',platform.processor() or platform.machine(), '| threads:',N_CPU_THREADS)
print('Mỗi seed dùng cùng split cố định; không dùng test để chọn hyperparameter.')

DATASET: /Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/datasets/breast_cancer_wisconsin.csv | shape: (569, 30) | classes: [212 357]
CPU: arm | threads: 4
Mỗi seed dùng cùng split cố định; không dùng test để chọn hyperparameter.


## 3. Chạy mô hình gốc trên CPU và xuất kết quả

In [3]:
from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
import importlib.metadata
rows=[]
for seed in SEEDS:
    for n in TRAIN_SIZES:
        xtr,ytr,xval,yval,xte,yte,ids,split=make_fold(seed,n)
        try:
            model=TabPFNClassifier.create_default_for_version(ModelVersion.V2,
                device='cpu',n_estimators=1,random_state=seed,
                fit_mode='low_memory',show_progress_bar=False)
            t=time.perf_counter();model.fit(xtr,ytr);fit_sec=time.perf_counter()-t
            t=time.perf_counter();prob=model.predict_proba(xte);pred_sec=time.perf_counter()-t
            pred=model.classes_[np.argmax(prob,axis=1)] if hasattr(model,'classes_') else np.argmax(prob,axis=1)
            rows.append(record('ok',seed,n,fit_sec,pred_sec,metrics(yte,pred,prob),version=importlib.metadata.version('tabpfn')))
        except Exception as e:
            rows.append(record('failed',seed,n,err=repr(e),version=importlib.metadata.version('tabpfn')))
            print(f'FAILED seed={seed}, n={n}:',repr(e))
file=save_results(rows)

/Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đã ghi: /Users/thanhpc02/Downloads/Tabular_4SOTA_CPU/results/tabpfnv2.csv


,model,dataset,seed,train_size,status,accuracy,macro_f1,roc_auc,log_loss,fit_seconds,predict_seconds,package_version,error
0,TabPFNv2,breast_cancer_wisconsin,42,50,ok,0.973684,0.971863,0.997024,0.078885,7.222088,0.148541,9.1.0,
1,TabPFNv2,breast_cancer_wisconsin,42,100,ok,0.956140,0.952638,0.996693,0.080244,0.008273,0.167266,9.1.0,


## 4. Ghi chú và giới hạn

CPU của TabPFN-2 bị giới hạn mẫu trong package chính thức; các mức 50/100/200/300 nằm trong phạm vi thử nghiệm. `n_estimators=1` chỉ phục vụ smoke test CPU, khác benchmark chính thức.